# Demo 4 - Observe resilient pool routing

## Scenario

Routing is a **per-call decision**, not a deployment-time choice. APIM selects by **priority, then weight, then health**. This demo uses two logical priority-1 members -- PTU East (weight 2) and PTU Central (weight 1) -- plus priority-2 PAYG spillover.

> **The mock proves the routing; the real AOAI backend proves the inference.** Routing mode uses an APIM-hosted mock origin so every response identifies its serving member. Inference mode sends the same client request through the same pool policy to real Azure OpenAI. `ptu-east` and `ptu-central` are logical names; no PTU capacity is required for the routing demonstration.


## Isolation

Demo 4 creates only `demo4-*` resources on the existing APIM instance used by Demos 1-3. Every ARM write is an idempotent PUT/upsert, so running this notebook twice does not duplicate resources.


In [ ]:
import json
import sys
import time
from collections import Counter

import requests

sys.path.append("..")
from shared import apim, auth, config, display

cfg = config.load_config(interactive=True)
config.validate_config(cfg)
cfg = config.ensure_resilient_pool_config(cfg, interactive=True)
config.validate_resilient_pool_config(cfg)

DEMO_API_ID = "demo4-resilient-pool-api"
DEMO_PRODUCT_ID = "demo4-resilient-pool"
DEMO_SUBSCRIPTION_ID = "demo4-resilient-pool-sub"
DEMO_PATH = "demo4-resilient-pool"
DEMO_NAMED_VALUE_AOAI_KEY = "demo4-aoai-key"
MOCK_API_ID = "demo4-mock-origin-api"
MOCK_SUBSCRIPTION_ID = "demo4-mock-origin-sub"
MOCK_PATH = "demo4-mock-origin"
POOL_ID = "demo4-aoai-pool"
API_STYLE = cfg.aoai_api_style
CIRCUIT_TRIP_SECONDS = 65
MOCK_MEMBER_IDS = {"east": "demo4-ptu-east", "central": "demo4-ptu-central", "payg": "demo4-payg"}
REAL_POOL_MEMBERS = [
    ("east", cfg.demo4_ptu_east_endpoint, 1, 2),
    ("central", cfg.demo4_ptu_central_endpoint, 1, 1),
    ("payg", cfg.demo4_payg_endpoint, 2, 1),
]

print("Demo 4 uses the existing APIM instance:", cfg.apim_name)
print("Logical priority-1 members: East (weight 2), Central (weight 1); PAYG is priority 2.")


## Preflight checks

The pool and circuit-breaker backend contract requires APIM Basic v2, Standard v2, Premium v2, or classic Standard/Premium. The mock API is hosted on this same APIM instance and creates no Azure resources outside APIM.


In [ ]:
display.header("Preflight checks")
service = apim.get_service(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
sku = service.get("sku", {}).get("name", "")
supported_skus = {"BasicV2", "StandardV2", "PremiumV2", "Standard", "Premium"}
_ = display.show_table([{
    "check": "APIM pool and breaker support",
    "status": "PASS" if sku in supported_skus else "FAIL",
    "detail": f"{cfg.apim_name}; SKU={sku or 'not returned'}",
    "remediation": "Use Basic v2, Standard v2, Premium v2, or classic Standard/Premium.",
}])
if sku not in supported_skus:
    raise RuntimeError("This APIM SKU does not support Demo 4 backend pools and circuit breakers.")


## Configure mock origin and resilient pool

The mock API has one operation per logical member. Its policy returns a small chat-completions-shaped response with `x-served-by`, or an origin `429` with `Retry-After` when its named-value fault switch is set. These switches are non-secret, idempotent APIM named values.


In [ ]:
display.header("Creating mock origin, backends, and pool")
GATEWAY_URL = apim.get_gateway_url(cfg.subscription_id, cfg.resource_group, cfg.apim_name).rstrip("/")
CIRCUIT_BREAKER = {
    "rules": [{
        "name": "demo4-throttle-and-server-errors",
        "failureCondition": {
            "count": 2,
            "interval": "PT1M",
            "statusCodeRanges": [{"min": 429, "max": 429}, {"min": 500, "max": 599}],
        },
        "tripDuration": "PT1M",
        "acceptRetryAfter": True,
    }]
}
for member in MOCK_MEMBER_IDS:
    apim.ensure_named_value(cfg.subscription_id, cfg.resource_group, cfg.apim_name,
                            f"demo4-mock-fault-{member}", f"demo4-mock-fault-{member}",
                            "healthy", secret=False)
    apim.ensure_named_value(cfg.subscription_id, cfg.resource_group, cfg.apim_name,
                            f"demo4-mock-retry-after-{member}", f"demo4-mock-retry-after-{member}",
                            str(CIRCUIT_TRIP_SECONDS), secret=False)

apim.ensure_api(cfg.subscription_id, cfg.resource_group, cfg.apim_name, MOCK_API_ID,
                "Demo 4 - Mock origin", MOCK_PATH, GATEWAY_URL, subscription_required=True)
apim.ensure_subscription(cfg.subscription_id, cfg.resource_group, cfg.apim_name, MOCK_SUBSCRIPTION_ID,
                         "Demo 4 Mock Origin Subscription", f"/apis/{MOCK_API_ID}")
MOCK_SUBSCRIPTION_KEY = apim.get_subscription_key(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name, MOCK_SUBSCRIPTION_ID)
apim.ensure_named_value(cfg.subscription_id, cfg.resource_group, cfg.apim_name,
                        "demo4-mock-origin-key", "demo4-mock-origin-key",
                        MOCK_SUBSCRIPTION_KEY, secret=True)
MOCK_BACKEND_CREDENTIALS = {"header": {"Ocp-Apim-Subscription-Key": ["{{demo4-mock-origin-key}}"]}}
for member in MOCK_MEMBER_IDS:
    apim.ensure_operation(cfg.subscription_id, cfg.resource_group, cfg.apim_name, MOCK_API_ID,
                          member, f"Demo 4 mock {member}", "POST", f"/{member}")
with open("../policies/demo4-mock-origin.xml", encoding="utf-8-sig") as f:
    apim.set_api_policy(cfg.subscription_id, cfg.resource_group, cfg.apim_name, MOCK_API_ID, f.read())

for member, backend_id in MOCK_MEMBER_IDS.items():
    apim.ensure_backend(cfg.subscription_id, cfg.resource_group, cfg.apim_name, backend_id,
                        f"{GATEWAY_URL}/{MOCK_PATH}/{member}",
                        description=f"Demo 4 mock pool member {backend_id}", circuit_breaker=CIRCUIT_BREAKER,
                        credentials=MOCK_BACKEND_CREDENTIALS)
apim.ensure_backend_pool(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name, POOL_ID,
    [{"id": MOCK_MEMBER_IDS[member], "priority": priority, "weight": weight}
     for member, _, priority, weight in REAL_POOL_MEMBERS],
    description="Demo 4 priority/weight resilient pool",
)
display.banner("Routing mode is ready: mock member responses expose x-served-by.", kind="success")


In [ ]:
display.header("Creating the client-facing API")
apim.ensure_api(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_API_ID,
                "Demo 4 - Resilient backend pool", DEMO_PATH, cfg.aoai_endpoint.rstrip("/"))
OPERATION_URL_TEMPLATE = "/openai/v1/chat/completions" if API_STYLE == "v1" else f"/openai/deployments/{cfg.aoai_deployment}/chat/completions"
apim.ensure_operation(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_API_ID,
                      "chat-completions", "Chat Completions", "POST", OPERATION_URL_TEMPLATE)
apim.ensure_product(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_PRODUCT_ID,
                    "Demo4-Resilient-Pool", "Isolated product for Demo 4 resilient routing.", True, "published")
apim.ensure_product_api_link(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_PRODUCT_ID, DEMO_API_ID)
apim.ensure_subscription(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_SUBSCRIPTION_ID,
                         "Demo 4 Resilient Pool Subscription", f"/products/{DEMO_PRODUCT_ID}")
if cfg.aoai_key:
    apim.ensure_named_value(cfg.subscription_id, cfg.resource_group, cfg.apim_name,
                            DEMO_NAMED_VALUE_AOAI_KEY, DEMO_NAMED_VALUE_AOAI_KEY,
                            cfg.aoai_key, secret=True)
with open("../policies/demo4-resilient-pool.xml", encoding="utf-8-sig") as f:
    policy_xml = f.read()
if cfg.aoai_key:
    policy_xml = policy_xml.replace(
        '<authentication-managed-identity resource="https://cognitiveservices.azure.com" />',
        '<set-header name="api-key" exists-action="override"><value>{{demo4-aoai-key}}</value></set-header>',
    )
apim.set_api_policy(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_API_ID, policy_xml)


## Fixed client request and mode switch

The client request below is printed once and is byte-identical in all four routing columns and the inference baseline: same URL, headers, and body. Only APIM's backend URLs change when selecting a mode. The client never learns or selects a member.


In [ ]:
SUBSCRIPTION_KEY = apim.get_subscription_key(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_SUBSCRIPTION_ID)
REQUEST_URL = f"{GATEWAY_URL}/{DEMO_PATH}{OPERATION_URL_TEMPLATE}"
REQUEST_HEADERS = {"Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY, "Content-Type": "application/json"}
REQUEST_BODY = {"messages": [{"role": "user", "content": "Reply with the word pool."}], "max_tokens": 40}
if API_STYLE == "v1":
    REQUEST_BODY["model"] = cfg.aoai_deployment
REQUEST_PARAMS = {} if API_STYLE == "v1" else {"api-version": cfg.aoai_api_version}
print(json.dumps({"url": REQUEST_URL, "headers": REQUEST_HEADERS, "params": REQUEST_PARAMS, "body": REQUEST_BODY}, indent=2))
display.banner("This is the unchanged client request for every phase. x-served-by is backend trace evidence, not a client routing input.", kind="info")

def call_pool():
    start = time.time()
    response = requests.post(REQUEST_URL, headers=REQUEST_HEADERS, params=REQUEST_PARAMS,
                             json=REQUEST_BODY, timeout=90)
    return {
        "status": response.status_code,
        "served_by": response.headers.get("x-served-by", "not returned"),
        "retry_after": response.headers.get("Retry-After", ""),
        "latency_ms": round((time.time() - start) * 1000, 1),
    }

def set_mock_member(member, state="healthy", retry_after=CIRCUIT_TRIP_SECONDS):
    if member not in MOCK_MEMBER_IDS:
        raise ValueError(f"Unknown member {member!r}; choose one of {sorted(MOCK_MEMBER_IDS)}.")
    if state not in {"healthy", "429"}:
        raise ValueError("Mock member state must be 'healthy' or '429'.")
    apim.ensure_named_value(cfg.subscription_id, cfg.resource_group, cfg.apim_name,
                            f"demo4-mock-fault-{member}", f"demo4-mock-fault-{member}",
                            state, secret=False)
    apim.ensure_named_value(cfg.subscription_id, cfg.resource_group, cfg.apim_name,
                            f"demo4-mock-retry-after-{member}", f"demo4-mock-retry-after-{member}",
                            str(retry_after), secret=False)

def fault_member(member, retry_after=CIRCUIT_TRIP_SECONDS):
    """Make one mock member return a reversible 429 with Retry-After."""
    set_mock_member(member, "429", retry_after)

def heal_member(member):
    """Return one mock member to its healthy response."""
    set_mock_member(member, "healthy")

def heal_all():
    """Return every mock member to its healthy response."""
    for member in MOCK_MEMBER_IDS:
        heal_member(member)

def configure_mode(mode):
    if mode == "routing":
        members = [(member, f"{GATEWAY_URL}/{MOCK_PATH}/{member}") for member in MOCK_MEMBER_IDS]
        heal_all()
    elif mode == "inference":
        members = [(member, endpoint) for member, endpoint, _, _ in REAL_POOL_MEMBERS]
    else:
        raise ValueError("mode must be 'routing' or 'inference'.")
    for member, endpoint in members:
        apim.ensure_backend(cfg.subscription_id, cfg.resource_group, cfg.apim_name,
                            MOCK_MEMBER_IDS[member], endpoint,
                            description=f"Demo 4 {mode} pool member {MOCK_MEMBER_IDS[member]}",
                            circuit_breaker=CIRCUIT_BREAKER,
                            credentials=MOCK_BACKEND_CREDENTIALS if mode == "routing" else None)
    display.banner(f"{mode.title()} mode configured. The client request remains unchanged.", kind="info")

def member_table(phase, results):
    by_member = {member_id: [] for member_id in MOCK_MEMBER_IDS.values()}
    for result in results:
        by_member.setdefault(result["served_by"], []).append(result)
    return [{"phase": phase, "member": member, "observed responses": len(rows),
             "statuses": ", ".join(str(row["status"]) for row in rows) or "not observed",
             "Retry-After": ", ".join(row["retry_after"] for row in rows if row["retry_after"]) or ""}
            for member, rows in by_member.items()]


## Inference mode -- real AOAI baseline

This one baseline call routes through the same `<set-backend-service backend-id="demo4-aoai-pool" />` line to the configured real Azure OpenAI origin. It proves real inference traffic uses the pool. It does not provide member identity because a real AOAI response does not emit the mock trace header.


In [ ]:
configure_mode("inference")
inference_baseline = call_pool()
_ = display.show_table([{"mode": "real AOAI inference", **inference_baseline}])
display.banner("A successful response proves inference mode. Switch back to routing mode for visible member selection.", kind="info")


## Routing mode -- CALL 1

Switch to mock members. The first unchanged client call displays the member selected by the pool in `x-served-by`. This is observed response evidence.


In [ ]:
configure_mode("routing")
call_1 = call_pool()
_ = display.show_table(member_table("CALL 1", [call_1]))


## Routing mode -- CALL 2 and observed weighting

The next unchanged client call shows the next routing decision. East has weight 2 and Central weight 1. Over 30 observed healthy calls, the distribution should be approximately 2:1, not an exact small-sample promise.


In [ ]:
call_2 = call_pool()
_ = display.show_table(member_table("CALL 2", [call_2]))
weighted_results = [call_pool() for _ in range(30)]
weighted_counts = Counter(result["served_by"] for result in weighted_results)
_ = display.show_table([{"member": member, "observed calls": weighted_counts.get(member, 0)}
                        for member in MOCK_MEMBER_IDS.values()])
display.banner("Counts above come from x-served-by. Approximate weighting is expected over a larger sample; do not infer an exact 2:1 from two calls.", kind="info")


## Routing mode -- FAULT

Fault East only with a real mock-origin `429` and `Retry-After`. The two 429 responses meet the breaker threshold; subsequent successful priority-1 traffic should be visibly served by Central. Then fault Central too to demonstrate that PAYG is spillover only after priority-1 is unavailable. All named-value faults are healed in `finally`.


In [ ]:
configure_mode("routing")
try:
    fault_member("east", retry_after=CIRCUIT_TRIP_SECONDS)
    east_fault_attempts = [call_pool() for _ in range(12)]
    east_429s = [result for result in east_fault_attempts
                 if result["served_by"] == "demo4-ptu-east" and result["status"] == 429]
    central_after_east = [call_pool() for _ in range(6)]
    _ = display.show_table(member_table("FAULT: East 429, Central receives", east_fault_attempts + central_after_east))
    if len(east_429s) < 2:
        display.banner("Fewer than two observed East 429 responses; do not claim its breaker opened. Rerun after confirming the mock fault switch.", kind="warning")

    fault_member("central", retry_after=CIRCUIT_TRIP_SECONDS)
    both_primary_attempts = [call_pool() for _ in range(12)]
    _ = display.show_table(member_table("FAULT: both priority-1 members, PAYG spillover", both_primary_attempts))
    if not any(result["served_by"] == "demo4-payg" and result["status"] == 200 for result in both_primary_attempts):
        display.banner("No observed PAYG spillover yet; do not infer it. Confirm both primary breakers have tripped and repeat the calls.", kind="warning")
finally:
    heal_all()
    display.banner("All mock fault switches were healed. Circuit breakers remain open until their observed Retry-After/trip duration expires.", kind="info")


## Routing mode -- RECOVER

The fault switches are healthy again. Wait for the breaker period to expire, then make observed calls. East returning in `x-served-by` proves it is back in rotation and sharing load.


In [ ]:
display.header("Waiting for breaker recovery")
time.sleep(CIRCUIT_TRIP_SECONDS + 5)
recovery_results = [call_pool() for _ in range(12)]
_ = display.show_table(member_table("RECOVER", recovery_results))
if not any(result["served_by"] == "demo4-ptu-east" and result["status"] == 200 for result in recovery_results):
    display.banner("East was not observed after the wait; do not claim recovery. Wait longer and repeat this cell.", kind="warning")
else:
    display.banner("East was observed healthy again. The unchanged client request saw only backend trace changes.", kind="success")


## Heal everything

Run this cell at any time to restore all mock origins to healthy responses. It does not fabricate a closed breaker; wait for `Retry-After` or `tripDuration` before expecting a previously open member to return.


In [ ]:
heal_all()
display.banner("All Demo 4 mock member fault switches are healthy.", kind="success")


## Summary

- **Per-call routing:** APIM evaluates priority, then weight, then health. Priority-2 PAYG is used only when priority-1 capacity is unavailable.
- **Two honest modes:** the mock proves routing decisions with `x-served-by`; the real AOAI baseline proves the pool also governs inference traffic.
- **Acceptance:** **no client change at all; the backend ID changes in traces -- resilience without touching application code.**
- **Consistency rule:** real pool members must use the same model and version, or routing can silently change model behavior.


## Reset / teardown notes

The following **optional, clearly gated** cell removes Demo 1-4 workshop artifacts only. It never deletes the APIM instance. Set `REMOVE_WORKSHOP_ARTIFACTS = True` only after the workshop.


In [ ]:
REMOVE_WORKSHOP_ARTIFACTS = False
if REMOVE_WORKSHOP_ARTIFACTS:
    resources = [
        "apis/demo2-metering-api/diagnostics/applicationinsights",
        "apis/demo1-openai-api", "apis/demo2-metering-api", "apis/demo3-content-safety-api",
        "apis/demo4-resilient-pool-api", "apis/demo4-mock-origin-api",
        "subscriptions/demo1-token-governance-sub", "subscriptions/demo2-metering-sub",
        "subscriptions/demo3-content-safety-sub", "subscriptions/demo4-resilient-pool-sub",
        "subscriptions/demo4-mock-origin-sub",
        "backends/demo1-openai-backend", "backends/demo2-openai-backend",
        "backends/demo3-openai-backend", "backends/demo3-content-safety-backend",
        "backends/demo4-aoai-pool", "backends/demo4-ptu-east", "backends/demo4-ptu-central", "backends/demo4-payg",
    ]
    named_values = [
        "demo1-tokens-per-minute", "demo1-daily-token-cap", "demo1-aoai-key", "demo2-aoai-key",
        "demo3-aoai-key", "demo3-content-safety-key", "demo4-aoai-key", "demo4-mock-origin-key",
        *[f"demo4-mock-fault-{member}" for member in MOCK_MEMBER_IDS],
        *[f"demo4-mock-retry-after-{member}" for member in MOCK_MEMBER_IDS],
    ]
    for resource in resources:
        apim.delete_apim_resource_if_exists(cfg.subscription_id, cfg.resource_group, cfg.apim_name, resource)
    for named_value in named_values:
        apim.delete_named_value_if_exists(cfg.subscription_id, cfg.resource_group, cfg.apim_name, named_value)
    display.banner("Demo 1-4 workshop artifacts removed; the APIM instance remains.", kind="success")
else:
    display.banner("Cleanup is disabled. Set REMOVE_WORKSHOP_ARTIFACTS = True only after the workshop.", kind="info")
